<a href="https://colab.research.google.com/github/YoungHyunKoo/UHI_SA_GEE/blob/main/MODIS_Heat_Island_census.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### *Python programming for Google Earth Engine*
# **Week 2-2. Compare MODIS temperature output with US Census data**

# **Import necessary libraries**

In [ ]:
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import datetime as dt
from tqdm import tqdm
from scipy import stats

# **Sociodemographic information**

In [ ]:
# Demographic data (race)
file = glob.glob("F:\\Heat_Island\\Census\\Demographic\\*-Data.csv")[0]
race = pd.read_csv(file)
race = race.loc[1:, ["GEO_ID", "DP05_0038PE", "DP05_0071PE", "DP05_0077PE"]].reset_index(drop = True)
race = race.rename(columns = {"DP05_0038PE": "black", "DP05_0071PE": "hispanic", "DP05_0077PE": "white"})
for i in range(0, len(race)):
    race.loc[i, "GEOID"] = str(race.loc[i, "GEO_ID"][9:])

demo = race.copy()
demo.head()

In [ ]:
# Demographic data (age)
file = glob.glob("F:\\Heat_Island\\Census\\Age_and_Sex\\*-Data.csv")[0]
age = pd.read_csv(file)
age = age.loc[1:, ["S0101_C02_030E"]].reset_index(drop = True)
demo["over65"] = age["S0101_C02_030E"].values
demo.head()

In [ ]:
# Demographic data (income)
file = glob.glob("F:\\Heat_Island\\Census\\income\\*-Data.csv")[0]
income = pd.read_csv(file)
income = income.loc[1:, ["S1901_C01_012E"]].reset_index(drop = True)
demo["income"] = income["S1901_C01_012E"].values
demo.head()

In [ ]:
# Demographic data (insurace)
file = glob.glob("F:\\Heat_Island\\Census\\Health_insurance\\*-Data.csv")[0]
health = pd.read_csv(file)
health = health.loc[1:, ["S2701_C03_001E"]].reset_index(drop = True)
demo["insurance"] = health["S2701_C03_001E"].values
demo.head()

In [ ]:
# Load temperature data

temp_day = pd.read_csv("F:\\Heat_Island\\LST_Day_1km_tract_2002-2022.csv", index_col = 0)
temp_night = pd.read_csv("F:\\Heat_Island\\LST_Night_1km_tract_2002-2022.csv", index_col = 0)
temp_day_diff = pd.read_csv("F:\\Heat_Island\\LST_Day_1km_increase_SA_2002-2022.csv", index_col = 0)
temp_night_diff = pd.read_csv("F:\\Heat_Island\\LST_Night_1km_increase_SA_2002-2022.csv", index_col = 0)
temp_LST = temp_day.join(temp_night.loc[:, temp_night.keys()[36:]])
temp_diff = temp_day_diff.loc[:, ["urban", "LST", "LST_sum", "LST_win"]].join(temp_night_diff.loc[:, ["LST", "LST_sum", "LST_win"]],
                                                                   lsuffix='_Day', rsuffix='_Night')
temp_data = temp_LST.join(temp_diff)
temp_data["GEOID"] = temp_data["GEOID"].astype(str)
data = pd.merge(temp_data, demo, on=["GEOID"])
data.to_csv("F:\\Heat_Island\\Census\\Tract_data.csv")

In [ ]:
from scipy import stats

data = pd.read_csv("F:\\Heat_Island\\Census\\Tract_data.csv", index_col = 0)
data = data.replace('-', "0")

In [ ]:
def estimate_coef(x, y):
    # number of observations/points
    n = np.size(x)

    # mean of x and y vector
    m_x = np.mean(x)
    m_y = np.mean(y)

    # calculating cross-deviation and deviation about x
    SS_xy = np.sum(y*x) - n*m_y*m_x
    SS_xx = np.sum(x*x) - n*m_x*m_x

    # calculating regression coefficients
    b_1 = SS_xy / SS_xx
    b_0 = m_y - b_1*m_x

    return (b_0, b_1)

field1 = 'LST_Day_1km_2021'
field2 = 'hispanic'

print(field2)
data = pd.read_csv("F:\\Heat_Island\\Census\\Tract_data.csv", index_col = 0)
data = data.replace('-', "0")
x1 = data[field1].astype(float) - 273.15
y1 = data[field2].astype(float)
x = np.arange(int(x1.min()), int(x1.max())+1)
fig, ax = plt.subplots(1,1, figsize = (3,3), dpi = 90, sharey = True)
# plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)
plt.subplots_adjust(wspace=0.1, hspace=0.0)

r, p = stats.pearsonr(x1, y1) # np.corrcoef(x1, y1)[0, 1]
ax.scatter(x1, y1, s=10)
b, a = estimate_coef(x1, y1)
ax.plot(x, a*x+b, c = "k", ls = "--")
ax.set_xlabel("Day Temperature (deg C)")
ax.set_ylabel("Hispanic population (%)")
# ax.set_title(, fontsize = 11)
# ax[0].set_xlim(0, 90)
# ax[0].set_ylim(0, 0.4)
# ax.legend()

xmin, xmax = plt.xlim()
ymin, ymax = plt.ylim()

ax.text(xmax - (xmax-xmin)*0.05, ymin + (ymax-ymin)*0.05, "R={0:.2f}\n(p={1:.2f})".format(r, p), ha = "right", fontsize = 12)
ax.grid(ls = ":", lw = 0.5)


In [ ]:
field1 = 'LST_Day_1km_2021'
field2 = 'black'

print(field2)
data = pd.read_csv("F:\\Heat_Island\\Census\\Tract_data.csv", index_col = 0)
data = data.replace('-', "0")
x1 = data[field1].astype(float) - 273.15
y1 = data[field2].astype(float)
x = np.arange(int(x1.min()), int(x1.max())+1)
fig, ax = plt.subplots(1,1, figsize = (3,3), dpi = 90, sharey = True)
# plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)
plt.subplots_adjust(wspace=0.1, hspace=0.0)

r, p = stats.pearsonr(x1, y1) # np.corrcoef(x1, y1)[0, 1]
ax.scatter(x1, y1, s=10)
b, a = estimate_coef(x1, y1)
ax.plot(x, a*x+b, c = "k", ls = "--")
ax.set_xlabel("Day Temperature (deg C)")
ax.set_ylabel("Black population (%)")
# ax.set_title(, fontsize = 11)
# ax[0].set_xlim(0, 90)
# ax[0].set_ylim(0, 0.4)
# ax.legend()

xmin, xmax = plt.xlim()
ymin, ymax = plt.ylim()

ax.text(xmax - (xmax-xmin)*0.05, ymax - (ymax-ymin)*0.2, "R={0:.2f}\n(p={1:.2f})".format(r, p), ha = "right", fontsize = 12)
ax.grid(ls = ":", lw = 0.5)


In [ ]:
field1 = 'LST_Day_1km_2021'
field2 = 'white'

print(field2)
data = pd.read_csv("F:\\Heat_Island\\Census\\Tract_data.csv", index_col = 0)
data = data.replace('-', "0")
x1 = data[field1].astype(float) - 273.15
y1 = data[field2].astype(float)
x = np.arange(int(x1.min()), int(x1.max())+1)
fig, ax = plt.subplots(1,1, figsize = (3,3), dpi = 90, sharey = True)
# plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)
plt.subplots_adjust(wspace=0.1, hspace=0.0)

r, p = stats.pearsonr(x1, y1) # np.corrcoef(x1, y1)[0, 1]
ax.scatter(x1, y1, s=10)
b, a = estimate_coef(x1, y1)
ax.plot(x, a*x+b, c = "k", ls = "--")
ax.set_xlabel("Day Temperature (deg C)")
ax.set_ylabel("White population (%)")
# ax.set_title(, fontsize = 11)
# ax[0].set_xlim(0, 90)
# ax[0].set_ylim(0, 0.4)
# ax.legend()

xmin, xmax = plt.xlim()
ymin, ymax = plt.ylim()

ax.text(xmax - (xmax-xmin)*0.05, ymax - (ymax-ymin)*0.2, "R={0:.2f}\n(p={1:.2f})".format(r, p), ha = "right", fontsize = 12)
ax.grid(ls = ":", lw = 0.5)

In [ ]:
field1 = 'LST_Day_1km_2021'
field2 = 'income'

print(field2)
data = pd.read_csv("F:\\Heat_Island\\Census\\Tract_data.csv", index_col = 0)
data = data.replace('-', "0").replace("250,000+", "250000")
x1 = data[field1].astype(float) - 273.15
y1 = data[field2].astype(float)/1000
x = np.arange(int(x1.min()), int(x1.max())+1)
fig, ax = plt.subplots(1,1, figsize = (3,3), dpi = 90, sharey = True)
# plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)
plt.subplots_adjust(wspace=0.1, hspace=0.0)

r, p = stats.pearsonr(x1, y1) # np.corrcoef(x1, y1)[0, 1]
ax.scatter(x1, y1, s=10)
b, a = estimate_coef(x1, y1)
ax.plot(x, a*x+b, c = "k", ls = "--")
ax.set_xlabel("Day Temperature (deg C)")
ax.set_ylabel("Median income (1,000 USD)")
# ax.set_title(, fontsize = 11)
# ax[0].set_xlim(0, 90)
# ax[0].set_ylim(0, 0.4)
# ax.legend()

xmin, xmax = plt.xlim()
ymin, ymax = plt.ylim()

ax.text(xmax - (xmax-xmin)*0.05, ymax - (ymax-ymin)*0.2, "R={0:.2f}\n(p={1:.2f})".format(r, p), ha = "right", fontsize = 12)
ax.grid(ls = ":", lw = 0.5)

In [ ]:
field1 = 'LST_Day_1km_2021'
field2 = 'over65'

print(field2)
data = pd.read_csv("F:\\Heat_Island\\Census\\Tract_data.csv", index_col = 0)
data = data.replace('-', "0").replace("250,000+", "250000")
x1 = data[field1].astype(float) - 273.15
y1 = data[field2].astype(float)
x = np.arange(int(x1.min()), int(x1.max())+1)
fig, ax = plt.subplots(1,1, figsize = (3,3), dpi = 90, sharey = True)
# plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)
plt.subplots_adjust(wspace=0.1, hspace=0.0)

r, p = stats.pearsonr(x1, y1) # np.corrcoef(x1, y1)[0, 1]
ax.scatter(x1, y1, s=10)
b, a = estimate_coef(x1, y1)
ax.plot(x, a*x+b, c = "k", ls = "--")
ax.set_xlabel("Day Temperature (deg C)")
ax.set_ylabel("Population over 65 (%)")
# ax.set_title(, fontsize = 11)
# ax[0].set_xlim(0, 90)
# ax[0].set_ylim(0, 0.4)
# ax.legend()

xmin, xmax = plt.xlim()
ymin, ymax = plt.ylim()

ax.text(xmax - (xmax-xmin)*0.05, ymax - (ymax-ymin)*0.2, "R={0:.2f}\n(p={1:.2f})".format(r, p), ha = "right", fontsize = 12)
ax.grid(ls = ":", lw = 0.5)

In [ ]:
field1 = 'LST_Day_1km_2021'
field2 = 'insurance'

print(field2)
data = pd.read_csv("F:\\Heat_Island\\Census\\Tract_data.csv", index_col = 0)
data = data.replace('-', "0").replace("250,000+", "250000")
x1 = data[field1].astype(float) - 273.15
y1 = data[field2].astype(float)
x = np.arange(int(x1.min()), int(x1.max())+1)
fig, ax = plt.subplots(1,1, figsize = (3,3), dpi = 90, sharey = True)
# plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)
plt.subplots_adjust(wspace=0.1, hspace=0.0)

r, p = stats.pearsonr(x1, y1) # np.corrcoef(x1, y1)[0, 1]
ax.scatter(x1, y1, s=10)
b, a = estimate_coef(x1, y1)
ax.plot(x, a*x+b, c = "k", ls = "--")
ax.set_xlabel("Day Temperature (deg C)")
ax.set_ylabel("Population with\nhealth insurance (%)")
# ax.set_title(, fontsize = 11)
# ax[0].set_xlim(0, 90)
ax.set_ylim(48, 102)
# ax.legend()

xmin, xmax = plt.xlim()
ymin, ymax = plt.ylim()

ax.text(xmin + (xmax-xmin)*0.05, ymin + (ymax-ymin)*0.05, "R={0:.2f}\n(p={1:.2f})".format(r, p), ha = "left", fontsize = 12)
ax.grid(ls = ":", lw = 0.5)

### Last update: 10/12/2022
### Credited by YoungHyun Koo (kooala317@gmail.com)